In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"
from diffusers import DiffusionPipeline, AutoencoderKL, UNet2DConditionModel
from diffusers.schedulers import DDIMScheduler
import numpy as np
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor, AutoModel
from accelerate import init_empty_weights, infer_auto_device_map, load_checkpoint_and_dispatch
import torch
import pdb
import copy
import sys
import argparse
import json
from tqdm import tqdm
import shortuuid
from blip3o.constants import *
from blip3o.conversation import conv_templates, SeparatorStyle
from blip3o.model.builder import load_pretrained_model
from blip3o.utils import disable_torch_init
from blip3o.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import math
import requests
from blip3o.conversation import conv_templates, SeparatorStyle
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
import base64
from io import BytesIO
# from qwen_vl_utils import process_vision_info
from blip3o.model import *

import re, random
import warnings

# model_path = sys.argv[1]
model_path = "/home/cvlab22/.cache/huggingface/hub/models--BLIP3o--BLIP3o-Model-8B/snapshots/c2edfc20814d4624c8d73ca3de351ebc3fa86508"
# model_path = "/home/cvlab12/models/BAGEL-7B-MoT"
diffusion_path = model_path + "/diffusion-decoder"

# model = blip3oQwenForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True, torch_dtype=torch.float16)


# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")


device_1 = 0

with warnings.catch_warnings():
    disable_torch_init()
    model_path = os.path.expanduser(model_path)
    model_name = get_model_name_from_path(model_path)
    tokenizer, multi_model, context_len = load_pretrained_model(model_path, None, model_name, device_map=None, device='cpu')

/home/cvlab22/anaconda3/envs/blip3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/cvlab22/anaconda3/envs/blip3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


OpenCLIP not installed


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
loading file vocab.json
loading file merges.txt
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file tokenizer.json
loading file chat_template.jinja
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file /home/cvlab22/.cache/huggingface/hub/models--BLIP3o--BLIP3o-Model-8B/snapshots/c2edfc20814d4624c8d73ca3de351ebc3fa86508/config.json
You are using a model of type blip3o_qwen to instantiate a model of type blip3o_qwen_inference. This is not supported for all configurations of models and can yield errors.
Model config b

 latent query size torch.Size([1, 64, 3584])


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Some weights of the model checkpoint at /home/cvlab22/.cache/huggingface/hub/models--BLIP3o--BLIP3o-Model-8B/snapshots/c2edfc20814d4624c8d73ca3de351ebc3fa86508 were not used when initializing blip3oQwenForInferenceLM: ['model.gen_vision_tower.vision_tower.model.blocks.0.attn.proj.bias', 'model.gen_vision_tower.vision_tower.model.blocks.0.attn.proj.weight', 'model.gen_vision_tower.vision_tower.model.blocks.0.attn.qkv.bias', 'model.gen_vision_tower.vision_tower.model.blocks.0.attn.qkv.weight', 'model.gen_vision_tower.vision_tower.model.blocks.0.mlp.fc1.bias', 'model.gen_vision_tower.vision_tower.model.blocks.0.mlp.fc1.weight', 'model.gen_vision_tower.vision_tower.model.blocks.0.mlp.fc2.bias', 'model.gen_vision_tower.vision_tower.model.blocks.0.mlp.fc2.weight', 'model.gen_vision_tower.vision_tower.model.blocks.0.norm1.bias', 'model.gen_vision_tower.vision_tower.model.blocks.0.norm1.weight', 'model.gen_vision_tower.vision_tower.model.blocks.0.norm2.bias', 'model.gen_vision_tower.vision_tow

In [3]:
multi_model

blip3oQwenForInferenceLM(
  (visual): Qwen2_5_VisionTransformerPretrainedModel(
    (patch_embed): Qwen2_5_VisionPatchEmbed(
      (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
    )
    (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-31): 32 x Qwen2_5_VLVisionBlock(
        (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
        (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
        (attn): Qwen2_5_VLVisionSdpaAttention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
        )
        (mlp): Qwen2_5_VLMLP(
          (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
          (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
          (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
          (act_fn): SiLU()
        )
      )
    )
    (merger): Qwen2_5_VLPatchMerger(
      (l

In [4]:
for n, p in multi_model.named_parameters():
    print(n, p.shape)

visual.patch_embed.proj.weight torch.Size([1280, 3, 2, 14, 14])
visual.blocks.0.norm1.weight torch.Size([1280])
visual.blocks.0.norm2.weight torch.Size([1280])
visual.blocks.0.attn.qkv.weight torch.Size([3840, 1280])
visual.blocks.0.attn.qkv.bias torch.Size([3840])
visual.blocks.0.attn.proj.weight torch.Size([1280, 1280])
visual.blocks.0.attn.proj.bias torch.Size([1280])
visual.blocks.0.mlp.gate_proj.weight torch.Size([3420, 1280])
visual.blocks.0.mlp.gate_proj.bias torch.Size([3420])
visual.blocks.0.mlp.up_proj.weight torch.Size([3420, 1280])
visual.blocks.0.mlp.up_proj.bias torch.Size([3420])
visual.blocks.0.mlp.down_proj.weight torch.Size([1280, 3420])
visual.blocks.0.mlp.down_proj.bias torch.Size([1280])
visual.blocks.1.norm1.weight torch.Size([1280])
visual.blocks.1.norm2.weight torch.Size([1280])
visual.blocks.1.attn.qkv.weight torch.Size([3840, 1280])
visual.blocks.1.attn.qkv.bias torch.Size([3840])
visual.blocks.1.attn.proj.weight torch.Size([1280, 1280])
visual.blocks.1.attn.p

# Load Und Weights

In [5]:
import torch
import sys
from safetensors.torch import load_file
import os
import peft
from peft import PeftConfig

peft_model_id = '/mnt/data1/dvlm/blip3o/checkpoints/counting_und/epoch0_step-500'
und_weight_path = os.path.join(peft_model_id, 'adapter_model.safetensors')

peft_config = PeftConfig.from_pretrained(
    peft_model_id,
)

ckpt = load_file(und_weight_path)   
new_ckpt = {}
for k in ckpt.keys():
    new_k = k.replace('base_model.model.model.language_model', 'model')
    new_ckpt[new_k] = ckpt[k]
    
    

multi_model.load_adapter(
    peft_config=peft_config,
    adapter_state_dict=new_ckpt,
    adapter_name='random'
    # peft_model_id
)

In [6]:
for n, p in multi_model.named_parameters():
    print(n, p.shape) if 'lora' in n else None

model.layers.0.self_attn.q_proj.lora_A.random.weight torch.Size([128, 3584])
model.layers.0.self_attn.q_proj.lora_B.random.weight torch.Size([3584, 128])
model.layers.0.self_attn.k_proj.lora_A.random.weight torch.Size([128, 3584])
model.layers.0.self_attn.k_proj.lora_B.random.weight torch.Size([512, 128])
model.layers.0.self_attn.v_proj.lora_A.random.weight torch.Size([128, 3584])
model.layers.0.self_attn.v_proj.lora_B.random.weight torch.Size([512, 128])
model.layers.0.self_attn.o_proj.lora_A.random.weight torch.Size([128, 3584])
model.layers.0.self_attn.o_proj.lora_B.random.weight torch.Size([3584, 128])
model.layers.0.mlp.gate_proj.lora_A.random.weight torch.Size([128, 3584])
model.layers.0.mlp.gate_proj.lora_B.random.weight torch.Size([18944, 128])
model.layers.0.mlp.up_proj.lora_A.random.weight torch.Size([128, 3584])
model.layers.0.mlp.up_proj.lora_B.random.weight torch.Size([18944, 128])
model.layers.0.mlp.down_proj.lora_A.random.weight torch.Size([128, 18944])
model.layers.0.ml